# 02 · Preparación de datos

Este notebook hace **una sola cosa**: limpiar los datos crudos, dividirlos en tres conjuntos
(train / validation / test) y guardarlos en `data/splits/`.

El resto de notebooks cargan directamente esos archivos; **nunca** tocan el CSV original.

---
### ¿Por qué tres conjuntos y no dos?
| Conjunto | Para qué sirve |
|---|---|
| **Train (70 %)** | El modelo aprende con estos datos |
| **Validation (15 %)** | Comparamos los modelos entre sí y ajustamos parámetros |
| **Test (15 %)** | Evaluación final, se mira **una sola vez** al terminar |

Con solo train/test se corre el riesgo de "contaminar" el test cada vez que se comparan modelos.

---
### ¿Por qué el preprocesado se aprende solo con train?
Si calculásemos la media para imputar valores nulos usando **todos** los datos,
estaríamos filtrando información del futuro (validation/test) al modelo — eso se llama
**data leakage** y hace que los resultados sean demasiado optimistas.

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Rutas
ROOT = Path('..').resolve()
RAW_CSV = ROOT / 'data' / 'raw' / 'airbnb-listings-extract.csv'
SPLITS_DIR = ROOT / 'data' / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 3
print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


## 1 · Carga y limpieza del CSV original

In [10]:
house_data = pd.read_csv(RAW_CSV, sep=None, engine='python', on_bad_lines='warn')
print(f'Dataset cargado: {house_data.shape[0]:,} filas, {house_data.shape[1]} columnas')

Dataset cargado: 14,780 filas, 89 columnas


In [11]:
# Columnas que no aportan información útil para predecir el precio
cols_to_drop = [
    'ID', 'Listing Url', 'Scrape ID', 'Last Scraped', 'Name', 'Summary', 'Space', 'Description',
    'Experiences Offered', 'Neighborhood Overview', 'Notes', 'Transit', 'Access', 'Interaction',
    'House Rules', 'Thumbnail Url', 'Medium Url', 'Picture Url', 'XL Picture Url', 'Host ID',
    'Host URL', 'Host Name', 'Host Since', 'Host Location', 'Host About', 'Host Response Time',
    'Host Response Rate', 'Host Acceptance Rate', 'Host Thumbnail Url', 'Host Picture Url',
    'Host Neighbourhood', 'Host Listings Count', 'Street', 'Smart Location', 'Country Code',
    'Calendar Updated', 'Has Availability', 'Calendar last Scraped', 'First Review', 'Last Review',
    'License', 'Jurisdiction Names', 'Calculated host listings count', 'Reviews per Month', 'Features',
    'Geolocation', 'Latitude', 'Longitude', 'Country', 'Host Verifications', 'Amenities',
    'Weekly Price', 'Monthly Price', 'Extra People', 'Neighbourhood', 'zipcode'
]

df = house_data.drop(columns=[c for c in cols_to_drop if c in house_data.columns])

# Convertir pies cuadrados a metros cuadrados
df['Square Meters'] = df['Square Feet'] * 0.092903
df = df.drop('Square Feet', axis=1)

# Filtrar solo Madrid y eliminar filas sin precio
df = df[df['City'] == 'Madrid'].copy()
df = df.dropna(subset=['Price'])

# Eliminar columnas ya innecesarias tras el filtro
df = df.drop(columns=['City', 'State'], errors='ignore')

print(f'Dataset Madrid limpio: {df.shape[0]:,} filas, {df.shape[1]} columnas')

Dataset Madrid limpio: 13,198 filas, 32 columnas


## 2 · División train / validation / test

Proporciones: **70 % train · 15 % validation · 15 % test**

El split se hace **antes** de cualquier transformación para evitar data leakage.

In [12]:
X = df.drop('Price', axis=1)
y = df['Price']

# Primer corte: 70 % train, 30 % temporal
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)

# Segundo corte: 50/50 del 30 % temporal → 15 % val, 15 % test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print(f'Train:      {len(X_train):>4} filas ({len(X_train)/len(X)*100:.0f} %)')
print(f'Validation: {len(X_val):>4} filas ({len(X_val)/len(X)*100:.0f} %)')
print(f'Test:       {len(X_test):>4} filas ({len(X_test)/len(X)*100:.0f} %)')

Train:      9238 filas (70 %)
Validation: 1980 filas (15 %)
Test:       1980 filas (15 %)


## 3 · Preprocesado (ajustado solo con train)

In [13]:
# ── Columnas por tipo ──────────────────────────────────────────────────────
cat_cols = [
    'Neighbourhood Cleansed', 'Neighbourhood Group Cleansed', 'Market',
    'Property Type', 'Room Type', 'Bed Type', 'Cancellation Policy'
]
cat_cols = [c for c in cat_cols if c in X_train.columns]

num_mean_cols = ['Bathrooms', 'Beds']
zero_cols = ['Host Total Listings Count', 'Security Deposit', 'Cleaning Fee',
             'Review Scores Rating', 'Bedrooms']
num_mean_cols = [c for c in num_mean_cols if c in X_train.columns]
zero_cols     = [c for c in zero_cols     if c in X_train.columns]

# ── Imputación numérica con media ──────────────────────────────────────────
imp_mean = SimpleImputer(strategy='mean')
for split in [X_train, X_val, X_test]:
    split[num_mean_cols] = imp_mean.fit_transform(X_train[num_mean_cols]) if split is X_train \
        else imp_mean.transform(split[num_mean_cols])

imp_mean.fit(X_train[num_mean_cols])
X_train[num_mean_cols] = imp_mean.transform(X_train[num_mean_cols])
X_val[num_mean_cols]   = imp_mean.transform(X_val[num_mean_cols])
X_test[num_mean_cols]  = imp_mean.transform(X_test[num_mean_cols])

# ── Imputación numérica con 0 ──────────────────────────────────────────────
imp_zero = SimpleImputer(strategy='constant', fill_value=0)
imp_zero.fit(X_train[zero_cols])
X_train[zero_cols] = imp_zero.transform(X_train[zero_cols])
X_val[zero_cols]   = imp_zero.transform(X_val[zero_cols])
X_test[zero_cols]  = imp_zero.transform(X_test[zero_cols])

# ── Zipcode → numérico ─────────────────────────────────────────────────────
for split in [X_train, X_val, X_test]:
    split['Zipcode'] = pd.to_numeric(
        split['Zipcode'].astype(str).str.strip(), errors='coerce'
    )

# ── Imputación categórica con moda ────────────────────────────────────────
imp_cat = SimpleImputer(strategy='most_frequent')
imp_cat.fit(X_train[cat_cols])
X_train[cat_cols] = imp_cat.transform(X_train[cat_cols])
X_val[cat_cols]   = imp_cat.transform(X_val[cat_cols])
X_test[cat_cols]  = imp_cat.transform(X_test[cat_cols])

print('Imputación completada.')

Imputación completada.


In [14]:
# ── One-Hot Encoding ──────────────────────────────────────────────────────
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)

for split, name in [(X_train, 'train'), (X_val, 'val'), (X_test, 'test')]:
    encoded = pd.DataFrame(
        ohe.transform(split[cat_cols]),
        columns=ohe_cols,
        index=split.index
    )
    split.drop(columns=cat_cols, inplace=True)
    split[ohe_cols] = encoded

print(f'One-Hot Encoding completado. Columnas totales: {X_train.shape[1]}')

One-Hot Encoding completado. Columnas totales: 210


In [15]:
# ── Imputación final (NaN residuales tras OHE) ────────────────────────────
imp_final = SimpleImputer(strategy='mean')
imp_final.fit(X_train)

X_train = pd.DataFrame(imp_final.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(imp_final.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test  = pd.DataFrame(imp_final.transform(X_test),  columns=X_test.columns,  index=X_test.index)

# ── Escalado estándar ─────────────────────────────────────────────────────
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

print('Escalado completado.')
print(f'Forma final → X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

Escalado completado.
Forma final → X_train: (9238, 210), X_val: (1980, 210), X_test: (1980, 210)


## 4 · Transformación logarítmica del precio y guardado

El precio tiene una distribución muy sesgada: la mayoría de alojamientos son baratos,
pero hay unos pocos muy caros que "estiran" la cola derecha. Esto hace que los modelos
lineales tengan dificultades para ajustarse bien.

Aplicamos `log1p` (logaritmo de precio + 1) antes de guardar. Esto "comprime" los precios
altos y hace la distribución más simétrica, lo que facilita el aprendizaje.

Al evaluar los modelos, deshacemos la transformación con `expm1` para volver a euros.

In [16]:
X_train.to_csv(SPLITS_DIR / 'X_train.csv', index=True)
X_val.to_csv(SPLITS_DIR   / 'X_val.csv',   index=True)
X_test.to_csv(SPLITS_DIR  / 'X_test.csv',  index=True)

# Guardamos el precio en escala logarítmica para mejorar el ajuste de los modelos
np.log1p(y_train).to_csv(SPLITS_DIR / 'y_train.csv', index=True, header=True)
np.log1p(y_val).to_csv(SPLITS_DIR   / 'y_val.csv',   index=True, header=True)
np.log1p(y_test).to_csv(SPLITS_DIR  / 'y_test.csv',  index=True, header=True)

print('Splits guardados en data/splits/:')
for f in sorted(SPLITS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<20} {size_kb:>8.1f} KB')
print()
print('Nota: los archivos y_*.csv contienen log1p(precio).')
print('Usa np.expm1() al interpretar predicciones para volver a euros.')

Splits guardados en data/splits/:
  X_test.csv             8473.8 KB
  X_train.csv           39520.1 KB
  X_val.csv              8474.9 KB
  y_test.csv               47.2 KB
  y_train.csv             220.2 KB
  y_val.csv                47.2 KB

Nota: los archivos y_*.csv contienen log1p(precio).
Usa np.expm1() al interpretar predicciones para volver a euros.
